# PromptTemplate in LangChain

## What is a PromptTemplate?
A **PromptTemplate** is a reusable text blueprint with **placeholders** (variables).
Instead of hardcoding prompts, you define a template once and fill variables dynamically.

```
Template : "Explain {topic} in {style} style."
Input    : topic="AI", style="simple"
Output   : "Explain AI in simple style."
```

## Types

| Type | Use Case |
|---|---|
| `PromptTemplate` | Plain text prompts (no roles) |
| `ChatPromptTemplate` | Chat-style with system/human/ai roles |
| `FewShotPromptTemplate` | Include examples to guide the model |

In [ ]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv

load_dotenv(override=True)

llm = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=300)

---
## Type 1: `PromptTemplate` — Plain Text
Use for simple single-turn prompts without chat roles.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["topic", "style"],
    template="Explain {topic} in {style} style."
)

template = PromptTemplate.from_template("Explain {topic} in {style} style.")

# Inspect what the prompt looks like BEFORE sending to LLM
filled = template.invoke({"topic": "AI", "style": "simple"})
print("Type :", type(filled).__name__)   # StringPromptValue
print("Value:", filled.text)             # the final string

In [ ]:
# Chain template with LLM and get response
chain = template | llm
response = chain.invoke({"topic": "AI", "style": "simple"})
print(response.content)

---
## Type 2: `ChatPromptTemplate` — With Roles
Best for chat models like Claude. Supports **system**, **human**, and **ai** roles.

| Role | Purpose |
|---|---|
| `system` | Sets model behaviour / persona |
| `human` | The user message |
| `ai` | Previous AI response (for conversation history) |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Answer in {language}."),
    ("human",  "{question}"),
])

# Inspect what ChatPromptTemplate produces BEFORE sending to LLM
filled = prompt.invoke({
    "role": "helpful Python tutor",
    "language": "English",
    "question": "What is a list comprehension?"
})

print("Type:", type(filled).__name__)   # ChatPromptValue
print()
for msg in filled.messages:            # list of BaseMessage objects
    print(f"[{msg.type.upper()}] {msg.content}")

In [ ]:
# Chain with LLM
chain = prompt | llm
response = chain.invoke({
    "role": "helpful Python tutor",
    "language": "English",
    "question": "What is a list comprehension?"
})
print(response.content)

---
## Type 3: `FewShotPromptTemplate` — With Examples
Provide input/output examples so the model learns the **pattern** from context.

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Format for each example
example_fmt = PromptTemplate.from_template("Word: {word} Antonym: {antonym}")

examples = [
    {"word": "happy",  "antonym": "sad"},
    {"word": "fast",   "antonym": "slow"},
    {"word": "bright", "antonym": "dark"},
]

few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_fmt,
    suffix="""Word: {word}
Antonym:",   # actual question appended at the end
    input_variables=["word"]""",
)

# Inspect the full prompt BEFORE sending to LLM
filled = few_shot.invoke({"word": "cold"})
print("--- Full Prompt Sent to LLM ---")
print(filled.text)

In [ ]:
chain = few_shot | llm
response = chain.invoke({"word": "cold"})
print("Answer:", response.content)

---
## What Does Each Template Return on `.invoke()`?

| Template | Returns | Access via |
|---|---|---|
| `PromptTemplate` | `StringPromptValue` | `.text` |
| `ChatPromptTemplate` | `ChatPromptValue` | `.messages` (list of BaseMessage) |
| `FewShotPromptTemplate` | `StringPromptValue` | `.text` |

## Summary

```
PromptTemplate        ->  plain string with {vars}        ->  simple tasks
ChatPromptTemplate    ->  system + human + ai messages    ->  best for Claude
FewShotPromptTemplate ->  examples + question             ->  pattern-guided tasks
```

> **Tip:** Always call `prompt.invoke(vars)` first to **inspect** the final prompt before sending to the LLM.